In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

current_path = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [current_path, *current_path.parents]
     if (p / "Scripts").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the repository.")
    
# Load data
# FIPS
fips_path = PROJECT_ROOT/"Data/Geography/FIPS/US states FIPS.csv"   # FIPS path
df_fips = pd.read_csv(fips_path)  # Load FIPS
df_fips["FIPS Code"] = df_fips["FIPS Code"].astype(str).str.zfill(2)    # Ensure string, add 0 to single digits

# MSAs
msa_shapefile_path = PROJECT_ROOT/"Data/Geography/CBSA_shapefile_2025/tl_2025_us_cbsa.shp"   # Shapefile path
gdf_msa = gpd.read_file(msa_shapefile_path) # Load MSA shapefile

gdf_msa = gdf_msa[gdf_msa["LSAD"] == 'M1'].reset_index(drop=True)   # Remove micro
gdf_msa['State_Abbr'] = gdf_msa['NAME'].str[-2:]    # Get state abbriviations
gdf_msa = gdf_msa[gdf_msa['State_Abbr'].isin(df_fips['Postal Abbr.'])].copy()   # Filter only US states
valid_geoid = gdf_msa["GEOID"].astype(str).tolist()  # GEOID list

# Lawyers
lawyers_path = PROJECT_ROOT/"Data/BrightData_Lawyers/BrightData_Lawyers_master_normalized_1overN.csv" # Lawyers data Path
lawyers_data = pd.read_csv(lawyers_path)   # Load data
lawyers_data.rename(columns={'CBSA': 'AREA', 'CBSA_Name': 'MSA'}, inplace=True) # Rename columns
lawyers_data["AREA"] = lawyers_data["AREA"].astype("Int64").astype(str)
lawyers_data = lawyers_data[lawyers_data["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

# Proxies
# Family lawyers
divorce_path = PROJECT_ROOT/"Data/Proxies/Family/ACSDT1Y2010.B12503-Data.csv" # Divorces data Path
divorce_data = pd.read_csv(divorce_path)   # Load data

divorce_data = divorce_data[['GEO_ID', 'NAME', 'B12503_010E']].copy()  # Take relevant columns
divorce_data.rename(columns={'GEO_ID': 'AREA', 'NAME': 'MSA', 'B12503_010E': 'Female_Divorces'}, inplace=True) # Rename columns

divorce_data = divorce_data.iloc[1:].reset_index(drop=True) # Remove first row

divorce_data['AREA'] = divorce_data['AREA'].str[-5:]    # Get GEOID
divorce_data["AREA"] = divorce_data["AREA"].astype(str) # GEOID as string

divorce_data = divorce_data[divorce_data["AREA"].isin(valid_geoid)]   # Take only MSAs from the list

divorce_data = divorce_data.merge(lawyers_data[['AREA', 'Family Law_normalized_1overN_count']], on=['AREA'], how='left')
divorce_data = divorce_data.dropna(subset=['Family Law_normalized_1overN_count'])
divorce_data = divorce_data.rename(columns={"Family Law_normalized_1overN_count": "Family"})
divorce_data = divorce_data.drop(columns=["MSA", "MSA_Name", "msa_name"], errors="ignore")

divorce_data.to_csv(PROJECT_ROOT/"Data/Proxies/Family/Family_Proxy_Normalized.csv", index=False)